In [19]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = '../datasets/rossmann-store-sales'
STORE_FILE = os.path.join(DATA_DIR, 'store.csv')
TRAIN_FILE = os.path.join(DATA_DIR, 'train.csv')

def date_from_week_year(row, year_col: str, week_col: str, day: int = 1):
    """
    Convert (year, week) to a date, handling missing values.
    Returns Monday of that ISO week by default.
    """

    if pd.isna(row[year_col]) or pd.isna(row[week_col]):
        out = pd.NaT  # Missing date

    else:
        date_str = f"{int(row[year_col])}-W{int(row[week_col]):02d}-{day}"
        out = pd.to_datetime(date_str, format="%G-W%V-%u")

    return out

def date_from_month_year(row, year_col: str, month_col: str, day: int = 1):
    """
    Convert (year, month) to a date, handling missing values.
    Returns Monday of that ISO week by default.
    """

    if pd.isna(row[year_col]) or pd.isna(row[month_col]):
        out = pd.NaT  # Missing date

    else:
        date_str = f"{int(row[year_col])}-{int(month_col):02d}-{day:02d}"
        out = pd.to_datetime(date_str, format="%G-W%V-%u")

    return out

def in_promo2(row, date_col: str, interval_col: str, start_promo_date_col: str):
    """
    Determine whether a given store is in an active Promo2 period for a specific date.

    This function checks, for a given row in a DataFrame, whether the date in `date_col`
    falls within the active Promo2 intervals defined in `interval_col`, and occurs on or after
    the store's `start_promo_date_col`. It safely handles missing values.
    """

    month = row[date_col].strftime("%b")

    if pd.isna(month) or pd.isna(row[interval_col]):
        out = False
    else:
        out = (row[date_col] >= row[start_promo_date_col]) & (month in row[interval_col])

    return out

In [20]:
store_df = pd.read_csv(STORE_FILE)
train_df = pd.read_csv(TRAIN_FILE)

train_df['Date'] = pd.to_datetime(train_df['Date'])

C:\Users\m_kal\AppData\Local\Temp\ipykernel_3844\3074264490.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(TRAIN_FILE)


In [21]:
""" Process store dataframe """
store_df['Promo2SinceDate'] = store_df.apply(date_from_week_year,
                                             args=('Promo2SinceYear', 'Promo2SinceWeek'),
                                             axis=1)
store_df['CompetitionSinceDate'] = store_df.apply(date_from_week_year,
                                                  args=('CompetitionOpenSinceYear', 'CompetitionOpenSinceMonth'),
                                                  axis=1)

store_df['LogCompetitionDistance'] = store_df['CompetitionDistance'].apply(np.log1p)

store_df.drop(['Promo2SinceYear', 'Promo2SinceWeek', 'CompetitionOpenSinceYear', 'CompetitionOpenSinceMonth', 'CompetitionDistance'],
              axis=1,
              inplace=True)

In [23]:
""" Process train dataframe """
train_df = train_df.merge(store_df, on='Store')

train_df['Promo2'] = train_df.apply(in_promo2, args=('Date', 'PromoInterval', 'Promo2SinceDate'), axis=1).astype(int)

train_df['Promo2SinceDays'] = train_df['Promo2'] * (train_df['Date'] - train_df['Promo2SinceDate']).dt.days.fillna(0).astype(int) 
train_df['LogPromo2SinceDays'] = train_df['Promo2SinceDays'].apply(np.log1p) * train_df['Promo2'] # Multiply with promo2 indicator to zero-out invalid num. days (i.e. no promotions)

train_df['CompetitionSinceDays'] = (train_df['Date'] - train_df['CompetitionSinceDate']).dt.days

train_df.drop(['PromoInterval', 'Promo2SinceDate', 'Promo2SinceDays', 'CompetitionSinceDate'], axis=1, inplace=True)

In [24]:
train_df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,Promo2,LogCompetitionDistance,LogPromo2SinceDays,CompetitionSinceDays
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,0,7.147559,0.000000,2713.0
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,1,6.347389,7.576097,3063.0
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,1,9.556126,7.365180,3420.0
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,0,6.431331,0.000000,2349.0
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,0,10.305982,0.000000,193.0


In [27]:
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'), index_col=0)

test_df

,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
Id,,,,,,,
1,1,4,2015-09-17,1.0,1,0,0
2,3,4,2015-09-17,1.0,1,0,0
3,7,4,2015-09-17,1.0,1,0,0
4,8,4,2015-09-17,1.0,1,0,0
5,9,4,2015-09-17,1.0,1,0,0
...,...,...,...,...,...,...,...
41084,1111,6,2015-08-01,1.0,0,0,0
41085,1112,6,2015-08-01,1.0,0,0,0
41086,1113,6,2015-08-01,1.0,0,0,0


In [ ]:
""" Process test dataframe """
# Apply the same processing to the test dataframe here and make convenience functions

In [19]:
""" Feature engineering from gpt 

import pandas as pd
import numpy as np

# Extract datetime features
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek

# Compute time since competitor opened
df['CompetitionOpenDate'] = pd.to_datetime(
    dict(year=df['CompetitionOpenSinceYear'].replace(0, np.nan),
         month=df['CompetitionOpenSinceMonth'].replace(0, np.nan),
         day=1),
    errors='coerce'
)
df['CompetitionOpenMonths'] = ((df['Date'] - df['CompetitionOpenDate'])/np.timedelta64(1, 'M')).fillna(0)
"""




" Feature engineering from gpt \n\nimport pandas as pd\nimport numpy as np\n\n# Extract datetime features\ndf['Date'] = pd.to_datetime(df['Date'])\ndf['Year'] = df['Date'].dt.year\ndf['Month'] = df['Date'].dt.month\ndf['Week'] = df['Date'].dt.isocalendar().week\ndf['Day'] = df['Date'].dt.day\ndf['DayOfWeek'] = df['Date'].dt.dayofweek\n\n# Compute time since competitor opened\ndf['CompetitionOpenDate'] = pd.to_datetime(\n    dict(year=df['CompetitionOpenSinceYear'].replace(0, np.nan),\n         month=df['CompetitionOpenSinceMonth'].replace(0, np.nan),\n         day=1),\n    errors='coerce'\n)\ndf['CompetitionOpenMonths'] = ((df['Date'] - df['CompetitionOpenDate'])/np.timedelta64(1, 'M')).fillna(0)\n"